In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

from housepriceprediction.config import DATA_RAW, RANDOM_STATE, TARGET
from housepriceprediction.modeling import model_pipeline, FINAL_LASSO_ALPHA

In [41]:
# Data: same reading options as in EDA; outliers removed from training only (see docs/preprocessing.md).
OUTLIER_LIVING_AREA = 4000  # sq ft, dataset author's recommendation

train_df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
train_df = train_df[train_df["GrLivArea"] <= OUTLIER_LIVING_AREA].reset_index(drop=True)

y = np.log1p(train_df[TARGET])
X = train_df.drop(columns=["Id", TARGET])
print(f"{len(X)} houses, {X.shape[1]} raw features")

1456 houses, 79 raw features


In [42]:
# Tools shared by every experiment below.
# Every model is evaluated with evaluate(), so every model uses the same `cv` folds.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results: dict[str, np.ndarray] = {}


def evaluate(name: str, pipeline: Pipeline, verbose: bool = False) -> np.ndarray:
    """5-fold CV RMSE on log prices; stores the 5 scores in `results[name]`."""
    rmse_scores = -cross_val_score(
        pipeline, X, y, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    results[name] = rmse_scores
    print(f"{name:<45} Mean RMSE: {rmse_scores.mean():.4f} (std: {rmse_scores.std():.4f})")
    if verbose:
        print(f"{'':<45} 5 fold scores: {np.round(rmse_scores, 4)}")
    return rmse_scores


def pick_best(mean_scores: dict[float, float]) -> float:
    """Grid value with the lowest mean RMSE; warns when it sits at an edge of the grid."""
    grid = list(mean_scores)
    best = min(grid, key=lambda value: mean_scores[value])
    if best in (grid[0], grid[-1]):
        print(f"WARNING: best value {best} is at the edge of the grid -> extend the grid.")
    return best

In [43]:
# 1. Baseline: always predicts the median log price, ignores every feature.
_ = evaluate("Baseline (median)", model_pipeline(DummyRegressor(strategy="median"), scale=False), verbose=True)

Baseline (median)                             Mean RMSE: 0.3970 (std: 0.0178)
                                              5 fold scores: [0.3977 0.4023 0.4024 0.4184 0.3642]


In [44]:
# Baseline: RMSE (log) ~0.397. Any real model must be clearly below this.

In [45]:
# 2. Ridge - grid 1
ridge_grid_1 = [0.1, 1, 3, 10, 30, 100, 300]
ridge_means: dict[float, float] = {}
for alpha in ridge_grid_1:
    ridge_means[alpha] = evaluate(f"Ridge (alpha={alpha})", model_pipeline(Ridge(alpha=alpha), scale=True)).mean()

Ridge (alpha=0.1)                             Mean RMSE: 0.1197 (std: 0.0041)
Ridge (alpha=1)                               Mean RMSE: 0.1195 (std: 0.0041)
Ridge (alpha=3)                               Mean RMSE: 0.1191 (std: 0.0041)
Ridge (alpha=10)                              Mean RMSE: 0.1182 (std: 0.0042)
Ridge (alpha=30)                              Mean RMSE: 0.1169 (std: 0.0046)
Ridge (alpha=100)                             Mean RMSE: 0.1152 (std: 0.0054)
Ridge (alpha=300)                             Mean RMSE: 0.1146 (std: 0.0066)


In [46]:
# Ridge - grid 2: the minimum of grid 1 was at its edge (300), so the grid is extended upwards.
ridge_grid_2 = [500, 1000, 2000, 5000]
for alpha in ridge_grid_2:
    ridge_means[alpha] = evaluate(f"Ridge (alpha={alpha})", model_pipeline(Ridge(alpha=alpha), scale=True)).mean()

RIDGE_ALPHA = pick_best(ridge_means)
print(f"Chosen Ridge alpha: {RIDGE_ALPHA}")

Ridge (alpha=500)                             Mean RMSE: 0.1156 (std: 0.0072)
Ridge (alpha=1000)                            Mean RMSE: 0.1198 (std: 0.0082)
Ridge (alpha=2000)                            Mean RMSE: 0.1293 (std: 0.0093)
Ridge (alpha=5000)                            Mean RMSE: 0.1558 (std: 0.0110)
Chosen Ridge alpha: 300


In [47]:
# Ridge - choosing alpha (5-fold CV, RMSE on log prices)
#
# Grid 1: [0.1 ... 300] -> error kept decreasing, minimum at the edge (300), so the grid was extended.
# Grid 2: [500 ... 5000] -> error rises again after 300 (U-shaped curve).
#
# Chosen: alpha = 300, Mean RMSE = 0.1146 (std 0.0066).
#   - The curve is flat between 100 and 500 (differences ~0.001, well below the
#     fold-to-fold std of ~0.006), so a finer grid would only fit noise.
#   - Small alpha (0.1-10): almost no change; the model is slightly unstable but not bad.
#   - Large alpha (1000+): the error grows quickly -> underfitting (coefficients pushed towards 0).
#   - Why a strong penalty helps: 200+ columns after one-hot, many of them correlated
#     (TotalSF / GrLivArea / TotalBsmtSF, GarageCars / GarageArea).
# Gain from tuning: 0.1182 (alpha=10) -> 0.1146 (alpha=300), ~3%.

In [48]:
# 3. Lasso (scaled features, like Ridge). max_iter raised because small alphas converge slowly.
lasso_grid = [0.0001, 0.0003, 0.001, 0.002, 0.003, 0.005, 0.01, 0.03]
lasso_means: dict[float, float] = {}
for alpha in lasso_grid:
    lasso_means[alpha] = evaluate(
        f"Lasso (alpha={alpha})", model_pipeline(Lasso(alpha=alpha, max_iter=50000), scale=True)
    ).mean()
LASSO_ALPHA = pick_best(lasso_means)
print(f"Chosen Lasso alpha: {LASSO_ALPHA}")
assert LASSO_ALPHA == FINAL_LASSO_ALPHA, (
    f"Grid chose {LASSO_ALPHA}, but modeling.py has {FINAL_LASSO_ALPHA} -> update modeling.py"
)

Lasso (alpha=0.0001)                          Mean RMSE: 0.1183 (std: 0.0042)
Lasso (alpha=0.0003)                          Mean RMSE: 0.1166 (std: 0.0042)
Lasso (alpha=0.001)                           Mean RMSE: 0.1135 (std: 0.0047)
Lasso (alpha=0.002)                           Mean RMSE: 0.1116 (std: 0.0051)
Lasso (alpha=0.003)                           Mean RMSE: 0.1109 (std: 0.0056)
Lasso (alpha=0.005)                           Mean RMSE: 0.1118 (std: 0.0063)
Lasso (alpha=0.01)                            Mean RMSE: 0.1175 (std: 0.0074)
Lasso (alpha=0.03)                            Mean RMSE: 0.1447 (std: 0.0119)
Chosen Lasso alpha: 0.003


In [49]:
# Lasso - choosing alpha (5-fold CV, RMSE on log prices, scaled features)
#
# Grid: [0.0001 ... 0.03] -> minimum inside the grid (U-shaped curve).
# Chosen: alpha = 0.003, Mean RMSE = 0.1109 (std 0.0056).
#   - Flat between 0.002 and 0.005 (0.1116 / 0.1109 / 0.1118): differences well below
#     the fold-to-fold std (~0.005), so a finer grid would only fit noise.
#   - Small alpha (0.0001): 0.1183, close to an almost unregularized linear model.
#   - alpha >= 0.01: useful columns are pushed to zero too -> error grows (0.1447 at 0.03).
#   - No ConvergenceWarning with scaled features.
#
# Lasso (0.1109) vs Ridge (0.1146): Lasso better by ~0.004, still below one std -> a small edge,
# not a decisive win.
#
# Note on the first attempt: the first Lasso grid had no StandardScaler and used cv=5
# (unshuffled folds) instead of the shared `cv`. Its scores (best 0.1113 at alpha=0.0003) were
# not comparable with the other models, and its ConvergenceWarnings came from the unscaled
# features (square feet next to 0/1 columns). Fixed by evaluating everything through
# evaluate() and model_pipeline(..., scale=True).

In [50]:
# 4. XGBoost - untuned starting point. No scaling: tree splits do not depend on feature scale.
_ = evaluate(
    "XGBoost (untuned)",
    model_pipeline(
        XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
        scale=False,
    ),
)

XGBoost (untuned)                             Mean RMSE: 0.1177 (std: 0.0072)


In [51]:
# XGBoost - step 1: max_depth (how many questions each tree asks in a row)
for max_depth in [1, 2, 3, 4, 5, 6]:
    evaluate(
        f"XGBoost (max_depth={max_depth})",
        model_pipeline(
            XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=max_depth, random_state=RANDOM_STATE),
            scale=False,
        ),
    )

XGBoost (max_depth=1)                         Mean RMSE: 0.1175 (std: 0.0058)
XGBoost (max_depth=2)                         Mean RMSE: 0.1162 (std: 0.0054)
XGBoost (max_depth=3)                         Mean RMSE: 0.1177 (std: 0.0072)
XGBoost (max_depth=4)                         Mean RMSE: 0.1196 (std: 0.0091)
XGBoost (max_depth=5)                         Mean RMSE: 0.1235 (std: 0.0110)
XGBoost (max_depth=6)                         Mean RMSE: 0.1272 (std: 0.0107)


In [52]:
# XGBoost - step 2: learning_rate x n_estimators, product kept ~50 (same "total amount of learning")
for learning_rate, n_estimators in [(0.1, 500), (0.05, 1000), (0.02, 2500), (0.01, 5000)]:
    evaluate(
        f"XGBoost (lr={learning_rate}, n={n_estimators})",
        model_pipeline(
            XGBRegressor(
                n_estimators=n_estimators, learning_rate=learning_rate, max_depth=2, random_state=RANDOM_STATE
            ),
            scale=False,
        ),
    )

XGBoost (lr=0.1, n=500)                       Mean RMSE: 0.1162 (std: 0.0046)
XGBoost (lr=0.05, n=1000)                     Mean RMSE: 0.1162 (std: 0.0054)
XGBoost (lr=0.02, n=2500)                     Mean RMSE: 0.1152 (std: 0.0064)
XGBoost (lr=0.01, n=5000)                     Mean RMSE: 0.1154 (std: 0.0062)


In [53]:
# XGBoost - step 3: subsample (each tree sees only a random part of the houses)
for subsample in [1.0, 0.8]:
    evaluate(
        f"XGBoost (subsample={subsample})",
        model_pipeline(
            XGBRegressor(
                n_estimators=1000, learning_rate=0.05, max_depth=2, subsample=subsample, random_state=RANDOM_STATE
            ),
            scale=False,
        ),
    )

XGBoost (subsample=1.0)                       Mean RMSE: 0.1162 (std: 0.0054)
XGBoost (subsample=0.8)                       Mean RMSE: 0.1144 (std: 0.0057)


In [54]:
# XGBoost - step 4: colsample_bytree (each tree sees only a random part of the columns)
for colsample in [1.0, 0.7, 0.5, 0.3]:
    evaluate(
        f"XGBoost (colsample_bytree={colsample})",
        model_pipeline(
            XGBRegressor(
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=2,
                subsample=0.8,
                colsample_bytree=colsample,
                random_state=RANDOM_STATE,
            ),
            scale=False,
        ),
    )

XGBoost (colsample_bytree=1.0)                Mean RMSE: 0.1144 (std: 0.0057)
XGBoost (colsample_bytree=0.7)                Mean RMSE: 0.1138 (std: 0.0060)
XGBoost (colsample_bytree=0.5)                Mean RMSE: 0.1137 (std: 0.0065)
XGBoost (colsample_bytree=0.3)                Mean RMSE: 0.1124 (std: 0.0050)


In [55]:
# XGBoost - tuning, one hyperparameter at a time (5-fold CV, RMSE on log prices)
#
# Untuned (max_depth=3, lr=0.05, 1000 trees): 0.1177
# 1. max_depth [1..6]: best 2 (0.1162). Deeper trees -> worse (0.1272 at 6: they learn noise);
#    depth 1 slightly worse (0.1175) -> a few pairwise interactions help, but only a little.
# 2. learning_rate x n_estimators (product kept ~50): 0.1152-0.1162 -> plateau, within noise.
#    Kept lr=0.05, 1000 trees (practically the same score, 2.5-5x faster than the smaller rates).
# 3. subsample=0.8: 0.1162 -> 0.1144.
# 4. colsample_bytree [1.0, 0.7, 0.5, 0.3]: 0.1144 -> 0.1124 (std 0.0050).
#    Chosen 0.3. Stopped tuning: further gains would be noise.
#
# Final: max_depth=2, lr=0.05, n_estimators=1000, subsample=0.8, colsample_bytree=0.3
#        Mean RMSE = 0.1124 (std 0.0050).
# Shallow trees winning -> after preprocessing, price is mostly a sum of independent effects,
# which is why the linear models are so hard to beat on this dataset.

In [56]:
# 5. Final comparison: out-of-fold predictions (every house predicted by a model that never saw it),
#    errors in dollars (MAE, RMSE) and on log prices.
final_models = {
    "Baseline (median)": model_pipeline(DummyRegressor(strategy="median"), scale=False),
    f"Ridge (alpha={RIDGE_ALPHA})": model_pipeline(Ridge(alpha=RIDGE_ALPHA), scale=True),
    f"Lasso (alpha={LASSO_ALPHA})": model_pipeline(Lasso(alpha=LASSO_ALPHA, max_iter=50000), scale=True),
    "XGBoost (tuned)": model_pipeline(
        XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=2,
            subsample=0.8,
            colsample_bytree=0.3,
            random_state=RANDOM_STATE,
        ),
        scale=False,
    ),
}

true_price = np.expm1(y)
oof_predictions: dict[str, np.ndarray] = {}  # reused in the error analysis (stage 4)
rows = []

for name, pipeline in final_models.items():
    pred_log = cross_val_predict(pipeline, X, y, cv=cv, n_jobs=-1)
    oof_predictions[name] = pred_log
    pred_price = np.expm1(pred_log)
    rows.append({
        "Model": name,
        "MAE ($)": mean_absolute_error(true_price, pred_price),
        "RMSE ($)": root_mean_squared_error(true_price, pred_price),
        "RMSE (log)": root_mean_squared_error(y, pred_log),
    })

comparison_df = pd.DataFrame(rows).set_index("Model")
display(comparison_df.round({"MAE ($)": 0, "RMSE ($)": 0, "RMSE (log)": 4}))

,MAE ($),RMSE ($),RMSE (log)
Model,,,
Baseline (median),55031.0,78697.0,0.3974
Ridge (alpha=300),13881.0,20618.0,0.1148
Lasso (alpha=0.003),13339.0,19990.0,0.1110
XGBoost (tuned),13440.0,20932.0,0.1125


In [57]:
# Final comparison - out-of-fold predictions, every model on the same folds
#
#                        MAE ($)   RMSE ($)   RMSE (log)
#   Baseline (median)     55,031     78,697     0.3974
#   Ridge (alpha=300)     13,881     20,618     0.1148
#   Lasso (alpha=0.003)   13,339     19,990     0.1110
#   XGBoost (tuned)       13,440     20,932     0.1125
#
# - Best model: Lasso, on all three metrics. Its gap to XGBoost (0.0015 on log) is far below the
#   fold-to-fold std (~0.005), so the two are tied in practice. Lasso is preferred: simpler,
#   faster to train, and its coefficients can be read directly (stage 4).
# - Typical error: ~13,300 $ per house (MAE), about 4x smaller than the baseline (55,000 $).
# - RMSE ($) is ~1.5x MAE ($) for every model -> a few houses with large errors inflate RMSE.
#   Finding them is the job of the error analysis (stage 4).
# - XGBoost beats Ridge on log RMSE but has the highest RMSE ($): its errors are relatively
#   larger on expensive houses, where a few percent means many dollars.
# - Check: RMSE (log) here matches the CV means above (0.1148 / 0.1110 / 0.1125 vs
#   0.1146 / 0.1109 / 0.1124) -> same pipelines, same folds.